## **Install Dependencies**

In [1]:
!pip install torch transformers datasets --quiet


## **Import Libraries**

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import numpy as np
import math, random


## **Create a Simple Text Dataset**

In [ ]:
text_data = """
The future belongs to those who believe in the beauty of their dreams.
Life is 10 percent what happens to us and 90 percent how we react to it.
Keep your face always toward the sunshine—and shadows will fall behind you.
The only limit to our realization of tomorrow is our doubts of today.
In the middle of difficulty lies opportunity.
"""

corpus = text_data.lower().replace("\n", " ")
print(corpus)


## **Tokenization (Character-level for simplicity)**

In [ ]:
# Create character-level vocabulary
chars = sorted(list(set(corpus)))
vocab_size = len(chars)

stoi = {c:i for i,c in enumerate(chars)}
itos = {i:c for c,i in stoi.items()}

def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return "".join([itos[i] for i in l])

encoded_data = torch.tensor(encode(corpus), dtype=torch.long)
vocab_size


## **Prepare Training Dataset**

In [ ]:
block_size = 64  # context length

def get_batch(batch_size=16):
    idx = torch.randint(len(encoded_data) - block_size - 1, (batch_size,))
    x = torch.stack([encoded_data[i:i+block_size] for i in idx])
    y = torch.stack([encoded_data[i+1:i+block_size+1] for i in idx])
    return x, y

xb, yb = get_batch()
xb.shape, yb.shape


## **Self-Attention Head**

In [6]:
class Head(nn.Module):
    def __init__(self, head_size, n_embd, block_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # Mask so model cannot look ahead
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)     # (B, T, hs)
        q = self.query(x)   # (B, T, hs)

        # compute attention scores
        att = q @ k.transpose(-2, -1) / math.sqrt(C)
        att = att.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        att = torch.softmax(att, dim=-1)

        # weighted sum of values
        v = self.value(x)
        out = att @ v
        return out


## **Multi-Head Attention**

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, n_embd, block_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, n_embd, block_size)
                                    for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)

    def forward(self, x):
        return self.proj(torch.cat([h(x) for h in self.heads], dim=-1))


## **Feedforward Layer**

In [8]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd)
        )
    def forward(self, x):
        return self.net(x)


## **Transformer Block**

In [9]:
class Block(nn.Module):
    def __init__(self, n_embd, num_heads, block_size):
        super().__init__()
        head_size = n_embd // num_heads
        self.sa = MultiHeadAttention(num_heads, head_size, n_embd, block_size)
        self.ff = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


## **Mini GPT Model**

In [10]:
class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.n_embd = 128

        self.token_embedding_table = nn.Embedding(vocab_size, self.n_embd)
        self.position_embedding_table = nn.Embedding(block_size, self.n_embd)

        self.blocks = nn.Sequential(
            Block(self.n_embd, num_heads=4, block_size=block_size),
            Block(self.n_embd, num_heads=4, block_size=block_size),
        )

        self.ln_f = nn.LayerNorm(self.n_embd)
        self.lm_head = nn.Linear(self.n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T))
        x = tok_emb + pos_emb

        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.lm_head(x)

        if targets is None:
            return logits

        logits = logits.view(B*T, -1)
        targets = targets.view(B*T)

        loss = nn.CrossEntropyLoss()(logits, targets)
        return logits, loss

model = MiniGPT()


## **Training Loop**

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=3e-4)

epochs = 300
for step in range(epochs):

    xb, yb = get_batch()

    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print(f"Step {step} | Loss: {loss.item():.4f}")


## **Generate New Text**

In [ ]:
def generate(model, start_text="the ", max_new_tokens=150):
    model.eval()
    idx = torch.tensor(encode(start_text), dtype=torch.long).unsqueeze(0)

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits = model(idx_cond)
        logits = logits[:, -1, :]

        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)

        idx = torch.cat((idx, next_id), dim=1)

    return decode(idx[0].tolist())

print(generate(model, "the "))


## **Evaluation - Measure Perplexity**

In [ ]:
def perplexity(model, data):
    with torch.no_grad():
        xb = data[:block_size].unsqueeze(0)
        yb = data[1:block_size+1].unsqueeze(0)
        _, loss = model(xb, yb)
    return torch.exp(loss)

print("Perplexity:", perplexity(model, encoded_data))
